In [1]:
%pwd

'c:\\Users\\manoj\\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS\\research'

LOAD PDF

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)

extracted_data = load_pdf_file(str(DATA_DIR))
print("Pages loaded:", len(extracted_data))

c:\Users\manoj\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\manoj\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS
Data folder: c:\Users\manoj\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS\data
Pages loaded: 637


CLEAN METADATA

In [3]:
from langchain_core.documents import Document


def filter_to_minimal_docs(docs):
    minimal_docs = [
        Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source")}
        )
        for doc in docs
    ]
    return minimal_docs


filter_data = filter_to_minimal_docs(extracted_data)
print("Filtered documents:", len(filter_data))

Filtered documents: 637


SPLIT INTO CHUNKS

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


text_chunks = text_split(filter_data)
print("Length of text chunks:", len(text_chunks))

Length of text chunks: 5859


DOWNLOAD HUGGINGFACE EMBEDDINGS

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings


def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings


embeddings = download_hugging_face_embeddings()
query_result = embeddings.embed_query("Hello world")
print("Embedding dimension:", len(query_result))

C:\Users\manoj\AppData\Local\Temp\ipykernel_25876\499396414.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embedding dimension: 384


LOAD API KEYS

In [6]:
from dotenv import load_dotenv
import os


load_dotenv()

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GEMINI_API_KEY = (
    os.environ.get("GOOGLE_API_KEY")
    or os.environ.get("GEMINI_API_KEY")
    or os.environ.get("Gemini_API_Key")
    or os.environ.get("Gemini API Key")
)

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if GEMINI_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

print("Pinecone key loaded:", bool(PINECONE_API_KEY))
print("OpenAI key loaded:", bool(OPENAI_API_KEY))
print("Gemini key loaded:", bool(GEMINI_API_KEY))

Pinecone key loaded: True
OpenAI key loaded: True
Gemini key loaded: True


CONNECT TO PINECONE AND CREATE INDEX

In [7]:
from pinecone import Pinecone, ServerlessSpec


pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created")
else:
    print("Index already exists")

index = pc.Index(index_name)

Index already exists


UPLOAD DATA TO PINECONE

In [8]:
from langchain_pinecone import PineconeVectorStore


# This guard prevents duplicate uploads if you run this notebook again.
stats = index.describe_index_stats()
if hasattr(stats, "total_vector_count"):
    vector_count = stats.total_vector_count
else:
    vector_count = stats.get("total_vector_count", 0)

if vector_count == 0:
    docsearch = PineconeVectorStore.from_documents(
        documents=text_chunks,
        index_name=index_name,
        embedding=embeddings
    )
    print("Data uploaded to Pinecone")
else:
    print(f"Pinecone already has {vector_count} vectors. Skipping upload.")

Pinecone already has 5859 vectors. Skipping upload.


LOAD EXISTING PINECONE INDEX

In [9]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

print("docsearch loaded:", docsearch)

docsearch loaded: <langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000002391F728EB0>


DIRECT PINECONE QUERY CHECK

In [10]:
QUESTION = "what is Acetaminophen?"
TOP_K = 3


def search_pinecone(question, top_k=3):
    query_vector = embeddings.embed_query(question)

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    matches = results.get("matches", [])

    print("Question:", question)
    print("Relevant chunks from Pinecone:", len(matches))

    for i, match in enumerate(matches, 1):
        metadata = match.get("metadata", {})
        text = metadata.get("text", "")
        source = metadata.get("source", "")

        print(f"\n--- Pinecone Chunk {i} ---")
        print("ID:", match.get("id"))
        print("Score:", match.get("score"))
        print("Source:", source)
        print("Text:")
        print(text[:1000])

    return matches


pinecone_matches = search_pinecone(QUESTION, TOP_K)

Question: what is Acetaminophen?
Relevant chunks from Pinecone: 3

--- Pinecone Chunk 1 ---
ID: e2f1db32-c235-4a6e-910c-b04ac0abc8b8
Score: 0.716267586
Source: c:\Users\manoj\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS\data\Medical_book.pdf
Text:
tion. 330 C Street SW, Washington, DC 20447. (800) 392-
3366.
OTHER
Elder Abuse Prevention. <http://www.oaktrees.org/elder>.
National Institute on Drug Abuse. <http://www.nida.nih.gov>.
Laith Farid Gulli, M.D.
Bilal Nasser, M.Sc.
Acceleration-deceleration cervical injury
see Whiplash
ACE inhibitors see Angiotensin-converting
enzyme inhibitors
Acetaminophen
Definition
Acetaminophen is a medicine used to relieve pain
and reduce fever.
Purpose
Acetaminophen is used to relieve many kinds of

--- Pinecone Chunk 2 ---
ID: 6866a67c-3731-47d4-8598-60678a7e611c
Score: 0.628770828
Source: c:\Users\manoj\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS\data\Medical_book.pdf
Text:
immediate medical attention.
Interactions
Acetaminophen ma

TEST RETRIEVAL

In [11]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

retrieved_docs = retriever.invoke(QUESTION)

print("Number of results:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:700])
    print(doc.metadata)

Number of results: 3

--- Result 1 ---
tion. 330 C Street SW, Washington, DC 20447. (800) 392-
3366.
OTHER
Elder Abuse Prevention. <http://www.oaktrees.org/elder>.
National Institute on Drug Abuse. <http://www.nida.nih.gov>.
Laith Farid Gulli, M.D.
Bilal Nasser, M.Sc.
Acceleration-deceleration cervical injury
see Whiplash
ACE inhibitors see Angiotensin-converting
enzyme inhibitors
Acetaminophen
Definition
Acetaminophen is a medicine used to relieve pain
and reduce fever.
Purpose
Acetaminophen is used to relieve many kinds of
{'source': 'c:\\Users\\manoj\\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS\\data\\Medical_book.pdf'}

--- Result 2 ---
immediate medical attention.
Interactions
Acetaminophen may interact with a variety of other
medicines. When this happens, the effects of one or both
of the drugs may change or the risk of side effects may
be greater. Among the drugs that may interact with
acetaminophen are alcohol, nonsteroidal anti-inflam-
matory drugs (NSAIDs) such as 

DIAGNOSTIC CHECK

In [12]:
print("CWD:", os.getcwd())
print("Project root:", PROJECT_ROOT)
print("Data folder exists:", DATA_DIR.exists())
print("Files in data:", os.listdir(DATA_DIR) if DATA_DIR.exists() else "NOT FOUND")
print("Pinecone index stats:", index.describe_index_stats())

CWD: c:\Users\manoj\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS\research
Project root: c:\Users\manoj\Medical_Chatbot_with_LLMs_LangChain_Pinecone_Flask_AWS
Data folder exists: True
Files in data: ['Medical_book.pdf']
Pinecone index stats: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5859}},
 'total_vector_count': 5859,
 'vector_type': 'dense'}


In [13]:
from google import genai

GEMINI_API_KEY = (
    os.getenv("GOOGLE_API_KEY")
    or os.getenv("GEMINI_API_KEY")
    or os.getenv("Gemini_API_Key")
)

if not GEMINI_API_KEY:
    raise ValueError("Gemini API key not found. Add GOOGLE_API_KEY to your .env file.")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")
print("Using Gemini model:", GEMINI_MODEL)

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

Using Gemini model: gemini-2.5-flash-lite


In [14]:
# We call Gemini directly here to avoid version conflicts between
# langchain-google-genai and the tutorial's LangChain version.

In [15]:
def answer_with_gemini(question, docs):
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a helpful medical assistant for answering questions about medical topics.
Use only the retrieved information below to answer the question.
If the answer is not in the retrieved information, say you don't know.
Keep the answer concise.

Retrieved information:
{context}

Question: {question}
"""

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )
    return response.text

In [16]:
# The retriever gets relevant chunks from Pinecone.
# Gemini then writes the final answer using those chunks.

In [ ]:
QUESTION = "what is periscope?"

docs = retriever.invoke(QUESTION)
answer = answer_with_gemini(QUESTION, docs)

print(answer)

I don't know.


: 